$\textbf{Из-за специфики Alembic не получилось поместить весь код в блокноте, поэтому ноутбук лучше не перезапускать}$

В процессе разработки всегда возникают разные версии кода. В частности, возникает проблема необходимость создания миграций БД, то есть кода, который переводит БД из одного состояния в другое. Миграции должны быть обратимы (возможность вернуться в предыдущее состояние) и упорядочены (должно быть понятно, в каком порядке применять изменения).

Самый простой инструмент для решения этой задачи - SQL. Код для перехода между версиями можно написать в в .sql файлах с помощью запросов CREATE TABLE, INSERT INTO, ALTER TABLE, DROP TABLE, и т.д. Однако у этого подхода есть ряд недостатков, а именно:
- Это небезопасно: отсутствует защита от выполнения изменений, не рассчитанных на текущую версию
- Много однотипного кода, при написании которого легко ошибиться
- Все sql скрипты нужно запускать вручную, при этом контролируя порядок запуска
- Текущее состояние БД определить невозможно, нужно запоминать всю историю изменений
- Не всегда есть доступ к машине, на которой находится база данных

Для решения этих проблем существует библиотека alembic, автоматизирующая процессы миграций БД

In [38]:
import sqlalchemy as sa
import sqlite3
import os
import pytest
# os.system("pip install alembic sqlalchemy")

In [ ]:
os.system("alembic init alembic")

Creating directory '/Users/grigorychaykovsky/Desktop/alembic/alembic' ...  done
Creating directory '/Users/grigorychaykovsky/Desktop/alembic/alembic/versions' ...  done
Generating /Users/grigorychaykovsky/Desktop/alembic/alembic/script.py.mako ...  done
Generating /Users/grigorychaykovsky/Desktop/alembic/alembic/env.py ...  done
Generating /Users/grigorychaykovsky/Desktop/alembic/alembic/README ...  done
Generating /Users/grigorychaykovsky/Desktop/alembic/alembic.ini ...  done
Please edit configuration/connection/logging settings in '/Users/grigorychaykovsky/Desktop/alembic/alembic.ini' before proceeding.


0

Теперь в файле ```alembic.ini``` нужно настроить sqlalchemy.url. В нашем случае
```sqlalchemy.url = sqlite+pysqlite:///users.db```

А в папке ```alembic``` находятся файл ```env.py``` и ```versions```.

In [39]:
from sqlalchemy import create_engine
url = 'sqlite+pysqlite:///users.db'
engine = create_engine(url, echo=True)


Теперь создадим первую миграцию - версию нашей базы данных - под названием "create t_users". В этой версии мы будем просто создавать таблицу

In [ ]:
os.system('alembic revision -m "create t_users"')

Generating /Users/grigorychaykovsky/Desktop/alembic/alembic/versions/ecdc8e0f4199_create_t_users.py ...  done


0

В директории alembic/versions появился первый файл, а именно ```ecdc8e0f4199_create_t_users.py```

ecdc8e0f4199 - это Revision Id, уникальный номер созданной версии, который можно будет использовать в команде ```alembic upgrade```

Попробуем сделать первый апгрейд - создадим базу данных ```t_users```. Для этого определим функции upgrade и downgrade в файле ```ecdc8e0f4199_create_t_users.py``` следующим образом:

```
def upgrade() -> None:
    op.create_table(
        "t_users",
        sa.Column("id", sa.Integer, primary_key=True),
        sa.Column("username", sa.String(50)),
        sa.Column("password", sa.String(50))
    )



def downgrade() -> None:
    op.drop_table("t_users")
```

Вместо сырых запросов используем модуль alembic.Operations, который позволяет писать меньше однотипного кода. Синтаксис позволяет лаконично написать запросы, которые на чистом sql выглядели бы сложнее

In [42]:
os.system("alembic upgrade head")

INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running upgrade  -> ecdc8e0f4199, create t_users
INFO  [alembic.runtime.migration] Running upgrade ecdc8e0f4199 -> 379a9326d79d, add user info
INFO  [alembic.runtime.migration] Running upgrade 379a9326d79d -> dd9b3618e64e, add playlists


0

У нас появилась таблица t_users! 

In [43]:
connection = sqlite3.connect('users.db')
cursor = connection.cursor()
print(cursor.execute('''
  PRAGMA table_info(t_users) 
''').fetchall())

[(0, 'id', 'INTEGER', 1, None, 1), (1, 'username', 'VARCHAR(50)', 0, None, 0), (2, 'password', 'VARCHAR(50)', 0, None, 0)]


Но в ней пока что ничего нет. Значит нужно это исправить, например создать новую версию, в которой добавятся какие-то данные. 

В файле с новой версией добавим такие функции перехода между версиями:

```
def upgrade() -> None:
    table = sa.sql.table(
        "t_users", 
        sa.sql.column("id", sa.Integer),
        sa.sql.column("username", sa.String(50)),
        sa.sql.column("password", sa.String(50))
    )
    op.bulk_insert(
        table,
        [
            {
                "id": 1,
                "username": "JustANickname",
                "password": "12345"
            },
            {
                "id": 2,
                "username": "StarGazer88",
                "password": "Moonlight!2023"
            },
            {
                "id": 3,
                "username": "BookLover91",
                "password": "ReadRelax2023"
            }
        ]
    )


def downgrade() -> None:
    op.execute('''DELETE FROM t_users''')
```

In [ ]:
os.system('alembic revision -m "add user info"')

Generating /Users/grigorychaykovsky/Desktop/alembic/alembic/versions/4e9937e2ad67_add_user_info.py ...  done


0

In [51]:
os.system('alembic upgrade head')

INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running upgrade ecdc8e0f4199 -> 379a9326d79d, add user info


0

In [57]:
print(cursor.execute('''
  SELECT * FROM t_users 
''').fetchall())

[(1, 'JustANickname', '12345'), (2, 'StarGazer88', 'Moonlight!2023'), (3, 'BookLover91', 'ReadRelax2023')]


Теперь в нашей таблице есть какие-то данные. Однако до сих пор мы никак не использовали потенциал alembic: создать базу данных и записать туда какие-то значения можно в несколько строчек кода и без него. 

Прелесть alembic заключается в том, что он позволяет быстро и удобно осуществлять переключения между последовательными версиями. Пока что у нас есть 3 версии: ```base```, где ничего нет, следующая за ней ```create_t_users```, и наконец ```add_user_info```

Напишем тесты, чтобы убедиться, что все работает как нужно. Для тестирования миграций можно использовать так называемый staircase method, который заключается в следующем: сначала тестируется переход от нулевой версии к первой и от первой к нулевой, затем от первой ко второй и наоборот, $\dots$, от (n-1)-й до n-й и от n-й к (n-1)-й. Если все они работают правильно, значит любой другой автоматный переход тоже будет работать правильно

In [ ]:
# проверим, в какой версии мы сейчас находимся
os.system('alembic current')

379a9326d79d (head)


INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


0

In [90]:
def down(N=1, base=False):
    if base:
        os.system('alembic downgrade base')
        return
    os.system(f'alembic downgrade -{N}')

def up(N=1, head=False):
    if head:
        os.system('alembic upgrade head')
        return
    os.system(f'alembic upgrade +{N}')

def whereami():
    os.system(f'alembic current')

In [94]:
def test0to1():
  down(base=True)
  up()

  assert not cursor.execute('''SELECT * FROM t_users''').fetchall()
  print(f"Переход к первой версии: ОК")




def test1to0():
  down(base=True)

  with pytest.raises(sqlite3.OperationalError, match='no such table: t_users'):
    print(cursor.execute('''
      SELECT * FROM t_users 
    ''').fetchall())

  print(f"Переход к нулевой версии: ОК")




def test1to2():
  down(base=True)
  up(N=2)

  assert cursor.execute('''SELECT * FROM t_users''').fetchall()

  print(f"Переход ко второй версии: ОК")


def test2to1():
  down(base=True)
  up(N=2)
  down()
  assert not cursor.execute('''SELECT * FROM t_users''').fetchall()

  print(f"Downgrade к первой версии: ОК")

test0to1()
test1to0()
test1to2()
test2to1()
print("ВСЕ РАБОТАЕТ")

INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running downgrade ecdc8e0f4199 -> , create t_users
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running upgrade  -> ecdc8e0f4199, create t_users


Переход к первой версии: ОК


INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running downgrade ecdc8e0f4199 -> , create t_users


Переход к нулевой версии: ОК


INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running upgrade  -> ecdc8e0f4199, create t_users
INFO  [alembic.runtime.migration] Running upgrade ecdc8e0f4199 -> 379a9326d79d, add user info


Переход ко второй версии: ОК


INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running downgrade 379a9326d79d -> ecdc8e0f4199, add user info
INFO  [alembic.runtime.migration] Running downgrade ecdc8e0f4199 -> , create t_users
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running upgrade  -> ecdc8e0f4199, create t_users
INFO  [alembic.runtime.migration] Running upgrade ecdc8e0f4199 -> 379a9326d79d, add user info


Downgrade к первой версии: ОК
ВСЕ РАБОТАЕТ


INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running downgrade 379a9326d79d -> ecdc8e0f4199, add user info


In [95]:
whereami()

ecdc8e0f4199


INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


Историю миграций можно увидеть с помощью специальной команды

Еще один мощный инструмент alembic - это возможность автоматической генерации миграции. Для этого в models.py запишем информацию о уже существующей таблице ```t_users```, а также о таблице ```t_playlists```, которую мы хотим добавить:

```
# models.py
from sqlalchemy import Column, String, Integer
from sqlalchemy.ext.declarative import declarative_base


Base = declarative_base()
metadata = Base.metadata

class User(Base):
    __tablename__ = "t_users"
    id = Column(Integer, primary_key=True)
    username = Column(String(50))
    password = Column(String(50))

class Playlist(Base):
    __tablename__ = 't_playlists'
    id = Column(Integer, primary_key=True)
    name = Column(String(50))
    genre = Column(String(50))
```

Чтобы alembic увидел, к чему нужно стремиться, нужно изменить поле ```target_metadata``` в ```env.py```:

```
# env.py
from models import metadata
target_metadata = metadata
```

In [89]:
os.system('alembic revision --autogenerate -m "add playlists"')

Generating /Users/grigorychaykovsky/Desktop/alembic/alembic/versions/dd9b3618e64e_add_playlists.py ...  done


INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.autogenerate.compare] Detected added table 't_playlists'


0

Появилась новая версия, в которой alembic сам добавил новую таблицу:

```
def upgrade() -> None:
    # ### commands auto generated by Alembic - please adjust! ###
    op.create_table('t_playlists',
    sa.Column('id', sa.Integer(), nullable=False),
    sa.Column('name', sa.String(length=50), nullable=True),
    sa.Column('genre', sa.String(length=50), nullable=True),
    sa.PrimaryKeyConstraint('id')
    )
    # ### end Alembic commands ###


def downgrade() -> None:
    # ### commands auto generated by Alembic - please adjust! ###
    op.drop_table('t_playlists')
    # ### end Alembic commands ###
```

Перекатимся на последнюю версию и напишем еще пару тестов, чтобы убедиться что alembic сделал все правильно

In [109]:
# в новой версии таблица появляется
def test2to3():
    down(base=True)
    up(N=3)

    assert not cursor.execute('SELECT * FROM t_playlists').fetchall()
    assert cursor.execute('PRAGMA table_info(t_playlists)').fetchall() == [
        (0, 'id', 'INTEGER', 1, None, 1), 
        (1, 'name', 'VARCHAR(50)', 0, None, 0), 
        (2, 'genre', 'VARCHAR(50)', 0, None, 0)
    ]
    print(f"2 to 3 OK")

# и при откате назад она удаляется
# в предыдущей версии такой таблицы нету
def test3to2():
    down(base=True)
    up(N=3)
    down()

    with pytest.raises(sqlite3.OperationalError, match='no such table: t_playlists'):
        cursor.execute('SELECT * FROM t_playlists')

    print(f"3 to 2 OK")

test2to3()
test3to2()


INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running downgrade 11504318a513 -> dd9b3618e64e, add column registered_at
INFO  [alembic.runtime.migration] Running downgrade dd9b3618e64e -> 379a9326d79d, add playlists
INFO  [alembic.runtime.migration] Running downgrade 379a9326d79d -> ecdc8e0f4199, add user info
INFO  [alembic.runtime.migration] Running downgrade ecdc8e0f4199 -> , create t_users
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running upgrade  -> ecdc8e0f4199, create t_users
INFO  [alembic.runtime.migration] Running upgrade ecdc8e0f4199 -> 379a9326d79d, add user info
INFO  [alembic.runtime.migration] Running upgrade 379a9326d79d -> dd9b3618e64e, add playlists


2 to 3 OK


INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running downgrade dd9b3618e64e -> 379a9326d79d, add playlists
INFO  [alembic.runtime.migration] Running downgrade 379a9326d79d -> ecdc8e0f4199, add user info
INFO  [alembic.runtime.migration] Running downgrade ecdc8e0f4199 -> , create t_users
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running upgrade  -> ecdc8e0f4199, create t_users
INFO  [alembic.runtime.migration] Running upgrade ecdc8e0f4199 -> 379a9326d79d, add user info
INFO  [alembic.runtime.migration] Running upgrade 379a9326d79d -> dd9b3618e64e, add playlists


3 to 2 OK


INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running downgrade dd9b3618e64e -> 379a9326d79d, add playlists


Попробуем выполнить еще одну вполне естественную задачу - добавить колонку в таблицу. Допустим, у нас поменялась логика приложения, и теперь для каждого пользователя нужно хранить время регистрации:

```
class User(Base):
    __tablename__ = "t_users"
    id = Column(Integer, primary_key=True)
    username = Column(String(50))
    password = Column(String(50))
    registered_at = Column(DateTime)
```

Однако если попробовать сгенерировать миграцию, возникнет ошибка. Дело в том, что команды по типу 
```op.add_column('t_users', sa.Column('registered_at', sa.DateTime(), nullable=True))``` запускают sql-запрос вида
```ALTER TABLE t_playlists ADD COLUMN creator DATETIME```, а sqlite такие запросы не поддерживает. Alembic поможет обойти и эту проблему тоже: специально для использования вместе с sqlite есть так называемый batch mode, который реализует copy-move подход с целью обойти ограничения на изменения таблиц. 

В ```env.py``` добавляем render_as_batch=True:
```
    context.configure(
        url=url,
        target_metadata=target_metadata,
        literal_binds=True,
        dialect_opts={"paramstyle": "named"},
        render_as_batch=True
    )
```
И снова генерируем миграцию

In [57]:
os.system('alembic revision --autogenerate -m "add column registered_at"')

Generating /Users/grigorychaykovsky/report/alembic/alembic/versions/11504318a513_add_column_registered_at.py ...  done


INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.autogenerate.compare] Detected added column 't_users.registered_at'


0

In [ ]:
os.system('alembic upgrade +1')

INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running upgrade dd9b3618e64e -> 11504318a513, add column registered_at


0

In [ ]:
# как обычно напишем тесты по staircase системе
os.system('alembic upgrade base+4')
def test3to4():
    res = cursor.execute('''PRAGMA table_info(t_users)''').fetchall()
    assert len(res) == 4
    assert res[3][1] == 'registered_at'
    assert res[3][2] == 'DATETIME'
    print('3 to 4 OK')
test3to4()
os.system('alembic downgrade -1')
def test4to3():
    res = cursor.execute('''PRAGMA table_info(t_users)''').fetchall()
    assert len(res) == 3
    print('4 to 3 OK')
test4to3()
os.system('alembic upgrade +1')


INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running upgrade dd9b3618e64e -> 11504318a513, add column registered_at


3 to 4 OK


INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running downgrade 11504318a513 -> dd9b3618e64e, add column registered_at


4 to 3 OK


INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running upgrade dd9b3618e64e -> 11504318a513, add column registered_at


0

Наконец, добавим в таблицу какую-нибудь связь. Например, для каждого плейлиста будем хранить user_id пользователя, который его создал.

```
class Playlist(Base):
    __tablename__ = 't_playlists'
    id = Column(Integer, primary_key=True)
    name = Column(String(50))
    genre = Column(String(50))
    creator = Column(Integer, ForeignKey(User.id))
```

Но добавление внешних ключей опять связано с особенностями SQLite. Если сейчас попробовать сгенерировать миграцию, мы получим ошибку ```ValueError: Constraint must have a name```. Хорошо, что SQLAlchemy позволяет не задумываться о названиях, передав шаблон naming_convention в метаданные нашей базы. Будем использовать стандарный naming_convention, который можно найти в [документации](https://alembic.sqlalchemy.org/en/latest/naming.html):

```
# models.py
convention = {
  "ix": "ix_%(column_0_label)s",
  "uq": "uq_%(table_name)s_%(column_0_name)s",
  "ck": "ck_%(table_name)s_%(constraint_name)s",
  "fk": "fk_%(table_name)s_%(column_0_name)s_%(referred_table_name)s",
  "pk": "pk_%(table_name)s"
}
Base.metadata = MetaData(naming_convention=convention)
```

И вот теперь ничего не мешает попросить сгенерировать нам миграцию

In [81]:
os.system('alembic revision --autogenerate -m "creators added"')

Generating /Users/grigorychaykovsky/report/alembic/alembic/versions/6c34fd9c3537_creators_added.py ...  done


INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.autogenerate.compare] Detected added column 't_playlists.creator'
INFO  [alembic.autogenerate.compare] Detected added foreign key (creator)(id) on table t_playlists


0

Миграция выглядит следующим образом:
```
def upgrade() -> None:
    # ### commands auto generated by Alembic - please adjust! ###
    with op.batch_alter_table('t_playlists', schema=None) as batch_op:
        batch_op.add_column(sa.Column('creator', sa.Integer(), nullable=True))
        batch_op.create_foreign_key(batch_op.f('fk_t_playlists_creator_t_users'), 't_users', ['creator'], ['id'])

    # ### end Alembic commands ###


def downgrade() -> None:
    # ### commands auto generated by Alembic - please adjust! ###
    with op.batch_alter_table('t_playlists', schema=None) as batch_op:
        batch_op.drop_constraint(batch_op.f('fk_t_playlists_creator_t_users'), type_='foreignkey')
        batch_op.drop_column('creator')

    # ### end Alembic commands ###
```
С помощью batch_alter_table удается избежать всех проблем, связанных с особенностями изменения таблиц в SQLite, а с помощью naming_convention - всех проблем, связанных с названиями объектов

In [ ]:
up()

INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running upgrade 11504318a513 -> 6c34fd9c3537, creators added


0

In [108]:
# и еще немного тестов
def test4to5():
    up(head=True)

    assert cursor.execute('''PRAGMA foreign_key_list(t_playlists)''').fetchall()

    print(f"4 to 5 OK")

def test5to4():
    up(head=True)
    down()

    assert not cursor.execute('''PRAGMA foreign_key_list(t_playlists)''').fetchall()

    print(f"5 to 4 OK")

test4to5()
test5to4()

INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


4 to 5 OK


INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


5 to 4 OK


INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running downgrade 6c34fd9c3537 -> 11504318a513, creators added


In [97]:
def history():
    os.system('alembic history')
history()

11504318a513 -> 6c34fd9c3537 (head), creators added
dd9b3618e64e -> 11504318a513, add column registered_at
379a9326d79d -> dd9b3618e64e, add playlists
ecdc8e0f4199 -> 379a9326d79d, add user info
<base> -> ecdc8e0f4199, create t_users
